# Tutorial 5: Advanced Domain Evaluation

Explore comfio's optional advanced domain modules:

1. Daylighting (Radiance-based)
2. Color Quality (CRI/CCT)
3. Reverberation (RT60)
4. Speech Intelligibility (STI)
5. Ventilation (CO₂ decay)
6. Psychrometrics

**Requires**: `pip install comfio[daylighting,color,acoustics,psychrometrics]`

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
n = 50

## 1. Daylighting Evaluation

Requires `pip install comfio[daylighting]`

In [ ]:
from comfio import evaluate_daylighting

# Spatial daylight autonomy metrics
daylight = evaluate_daylighting(
    illuminance=rng.normal(300.0, 100.0, n),
    target_illuminance=500.0,
    hours_of_operation=8.0,
)
print(f'Daylighting score: {np.mean(daylight.score):.1f}/100')
print(f'sDA: {daylight.sda:.1%}')

## 2. Color Quality Evaluation

Requires `pip install comfio[color]`

In [ ]:
from comfio import evaluate_color_quality

color = evaluate_color_quality(
    spectral_power=rng.uniform(0, 1, (n, 81)),  # 81 wavelengths (380-780nm)
    wavelengths=np.linspace(380, 780, 81),
)
print(f'CRI: {np.mean(color.cri):.1f}')
print(f'CCT: {np.mean(color.cct):.0f} K')
print(f'Color quality score: {np.mean(color.score):.1f}/100')

## 3. Reverberation Time (RT60)

In [ ]:
from comfio import evaluate_reverberation

reverb = evaluate_reverberation(
    room_volume=50.0,  # m³
    n_occupants=5,
    absorption_coeffs=np.array([0.2, 0.3, 0.4, 0.5, 0.6, 0.7]),
)
print(f'RT60: {reverb.rt60:.2f} s')
print(f'Reverberation score: {reverb.score:.1f}/100')

## 4. Speech Intelligibility (STI)

In [ ]:
from comfio import evaluate_speech_intelligibility

sti = evaluate_speech_intelligibility(
    impulse_response=rng.normal(0, 0.1, 48000),  # 1s at 48kHz
    sample_rate=48000,
    noise_level=35.0,  # dB background noise
)
print(f'STI: {sti.sti:.2f}')
print(f'Speech intelligibility score: {sti.score:.1f}/100')

## 5. Ventilation Rate from CO₂ Decay

Requires `pip install comfio[psychrometrics]`

In [ ]:
from comfio import evaluate_ventilation

vent = evaluate_ventilation(
    co2=rng.normal(800.0, 100.0, n),
    co2_outdoor=420.0,
    room_volume=50.0,
    n_occupants=5,
)
print(f'Ventilation rate: {vent.ventilation_rate:.1f} L/s/person')
print(f'Ventilation score: {np.mean(vent.score):.1f}/100')

## 6. Psychrometric Properties

In [ ]:
from comfio import get_psychrometrics

psych = get_psychrometrics(
    tdb=rng.normal(24.0, 2.0, n),
    rh=rng.normal(50.0, 5.0, n),
    pressure=101325.0,
)
print(f'Dew point: {np.mean(psych.dew_point):.1f}°C')
print(f'Wet bulb: {np.mean(psych.wet_bulb):.1f}°C')
print(f'Enthalpy: {np.mean(psych.enthalpy):.1f} kJ/kg')
print(f'Humidity ratio: {np.mean(psych.humidity_ratio):.4f} kg/kg')

## 7. Integration with Global IEQ

In [ ]:
from comfio import (
    evaluate_thermal, evaluate_visual, evaluate_acoustic, evaluate_iaq,
    calculate_global_ieq,
)

thermal = evaluate_thermal(
    tdb=rng.normal(24.0, 2.0, n),
    tr=rng.normal(24.0, 1.5, n),
    vr=np.full(n, 0.1),
    rh=rng.normal(50.0, 5.0, n),
    met=1.2, clo=0.5,
)
visual = evaluate_visual(illuminance=rng.normal(500.0, 50.0, n))
acoustic = evaluate_acoustic(laeq=rng.normal(40.0, 5.0, n))
iaq = evaluate_iaq(co2=rng.normal(800.0, 100.0, n))

ieq = calculate_global_ieq(
    thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq,
    daylighting=daylight,
    reverberation=reverb,
    speech_intelligibility=sti,
    ventilation=vent,
)
print(f'Global IEQ (with advanced domains): {np.mean(ieq.index):.1f}/100')
print(f'Domains: {ieq.domains}')